# Retrieval crowd vs. AI crowd: a running-shoe buying assistant

We build the same shopping assistant three ways:

- **Path A:** LLM only. Stuff the catalog into the prompt and let the model answer.
- **Path B:** Retrieval only. Hybrid (BM25 + semantic) RRF over the catalog.
- **Path C:** Collaboration. Elastic Agent Builder calls retrieval as a tool and the LLM finishes.

We close with the `named_queries` pro tip: a deterministic gate that lets retrieval short circuit the LLM call when it is confident.

## Setup

In [ ]:
%pip install -qU elasticsearch python-dotenv requests

In [ ]:
from dotenv import load_dotenv
import os
import json
import requests

load_dotenv()

ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
KIBANA_URL = os.getenv("KIBANA_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Connect to Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL, api_key=ELASTICSEARCH_API_KEY, request_timeout=30
)
es_client.info()

## Inference endpoints

Two roles. The embedding endpoint `.jina-embeddings-v5-text-small` is preconfigured by Elastic Inference Service, so we reference it directly in the index mapping with no PUT call. We create a chat completion endpoint backed by OpenAI.

| Role | Inference id | Service |
| :--- | :--- | :--- |
| Embeddings on `description_semantic` | `.jina-embeddings-v5-text-small` (preconfigured) | EIS |
| Chat completion | `shoes-chat` | OpenAI (`gpt-4o-mini`) |

In [ ]:
INFERENCE_ID = "shoes-chat"

try:
    es_client.inference.delete(inference_id=INFERENCE_ID)
except Exception:
    pass

es_client.inference.put(
    inference_id=INFERENCE_ID,
    task_type="chat_completion",
    body={
        "service": "openai",
        "service_settings": {
            "api_key": OPENAI_API_KEY,
            "model_id": "gpt-4o-mini",
        },
    },
)

print("Inference endpoint ready")

## Create the index

`description` is copied into a `semantic_text` field. `sku`, `surface`, `stability`, and `category` stay as `keyword` so they are queryable exactly. `name` stays as plain `text` for BM25 and constrained `match` clauses.

In [ ]:
INDEX_NAME = "shoes"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "sku": {"type": "keyword"},
            "name": {"type": "text"},
            "aliases": {"type": "keyword"},
            "description": {
                "type": "text",
                "copy_to": "description_semantic",
            },
            "description_semantic": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
            "surface": {"type": "keyword"},
            "stability": {"type": "keyword"},
            "category": {"type": "keyword"},
        }
    },
)
print(f"Created index: {INDEX_NAME}")

## Bulk load the catalog

A small synthetic catalog (~20 running shoes) loaded from `dataset.json`. The catalog covers road and trail, neutral and stability, daily trainers, tempo, race, and trail categories.

In [ ]:
from elasticsearch import helpers

with open("dataset.json") as f:
    shoes = json.load(f)

actions = ({"_index": INDEX_NAME, "_id": s["sku"], "_source": s} for s in shoes)
success, _ = helpers.bulk(es_client, actions, refresh=True)
print(f"Indexed {success} shoes")

## Test queries

Two shapes the assistant has to handle:

- **Identification.** Resolve to one SKU.
- **Recommendation.** Return one shoe with a short reason.

In [ ]:
Q_IDENTIFY = "Do you carry the Vaporfly 3 in men's 10, black?"
Q_RECOMMEND = (
    "a shoe for marathon training on asphalt, I mildly overpronate, "
    "prefer firmer cushioning"
)

## Path A: let the AI crowd win

Stuff the whole catalog into the prompt and ask the LLM to pick. Works at this size, breaks at 20,000 SKUs.

In [ ]:
def eis_chat(prompt: str) -> str:
    r = requests.post(
        f"{ELASTICSEARCH_URL}/_inference/chat_completion/shoes-chat/_stream",
        headers={
            "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
            "Content-Type": "application/json",
        },
        json={"messages": [{"role": "user", "content": prompt}]},
    )
    r.raise_for_status()

    text = ""
    for line in r.text.splitlines():
        if not line.startswith("data:"):
            continue
        chunk = line[len("data:") :].strip()
        if not chunk or chunk == "[DONE]":
            continue
        data = json.loads(chunk)
        for choice in data.get("choices", []):
            delta = choice.get("delta", {}).get("content")
            if delta:
                text += delta
    return text


def answer_llm_only(user_query: str) -> dict:
    catalog = [
        hit["_source"]
        for hit in es_client.search(
            index=INDEX_NAME,
            size=100,
            query={"match_all": {}},
            source_excludes=["description_semantic"],
        )["hits"]["hits"]
    ]

    prompt = f"""You are a running-shoe buying assistant.

Catalog (JSON):
{json.dumps(catalog, indent=2)}

User query: {user_query}

If the user is asking to identify a specific shoe, return JSON:
  {{"intent": "identify", "sku": "..."}}
If the user is asking for a recommendation, return JSON:
  {{"intent": "recommend", "sku": "...", "reason": "..."}}

Return ONLY the JSON object."""

    raw = eis_chat(prompt)
    cleaned = (
        raw.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    return json.loads(cleaned)


print("identify:", answer_llm_only(Q_IDENTIFY))
print("recommend:", answer_llm_only(Q_RECOMMEND))

## Path B: let the retrieval crowd win

Hybrid RRF over a BM25 `match` on `name` and a `semantic` retriever on `description_semantic`, then a reranker on top. The reranker uses the preconfigured `.rerank-v1-elasticsearch` endpoint from EIS. No LLM.

In [ ]:
def answer_retrieval_only(user_query: str) -> dict:
    body = {
        "retriever": {
            "text_similarity_reranker": {
                "retriever": {
                    "rrf": {
                        "retrievers": [
                            {"standard": {"query": {"match": {"name": user_query}}}},
                            {
                                "standard": {
                                    "query": {
                                        "semantic": {
                                            "field": "description_semantic",
                                            "query": user_query,
                                        }
                                    }
                                }
                            },
                        ],
                        "rank_window_size": 20,
                    }
                },
                "field": "description",
                "inference_id": ".rerank-v1-elasticsearch",
                "inference_text": user_query,
                "rank_window_size": 10,
            }
        },
        "size": 5,
        "_source": ["sku", "name"],
    }

    hits = es_client.search(index=INDEX_NAME, body=body)["hits"]["hits"]
    return {"intent": "list", "hits": [h["_source"] for h in hits]}


print("identify:", json.dumps(answer_retrieval_only(Q_IDENTIFY), indent=2))
print("recommend:", json.dumps(answer_retrieval_only(Q_RECOMMEND), indent=2))

## Path C: the collaboration with Agent Builder

We declare two tools and register an agent that pairs them with `shoes-chat`:

- `index_search` — free-text search over the catalog. Good for open-ended queries where the user does not specify exact attributes.
- `esql` — a parameterised ES|QL template with typed placeholders the agent fills from the user's query. This is the managed version of the "LLM as query builder" pattern: the agent translates *"a road shoe with stability support"* into `surface == "road"` and `stability == "stability"`, and the query engine runs the filter.

In [ ]:
headers = {
    "Authorization": f"ApiKey {ELASTICSEARCH_API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}

In [ ]:
TOOL_ID = "search_shoes"
ESQL_TOOL_ID = "filter_shoes"
AGENT_ID = "shoe-assistant"

requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{TOOL_ID}", headers=headers)
requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{ESQL_TOOL_ID}", headers=headers)

# Tool 1: free-text search over the shoes catalog.
requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json={
        "id": TOOL_ID,
        "type": "index_search",
        "description": (
            "Search the shoes catalog by free text. Returns up to 5 ranked "
            "candidates with sku, name, description, surface, stability, "
            "and category."
        ),
        "configuration": {"pattern": INDEX_NAME, "row_limit": 5},
    },
)
print("index_search tool created")

# Tool 2: parameterised ES|QL filter over structured attributes.
requests.post(
    f"{KIBANA_URL}/api/agent_builder/tools",
    headers=headers,
    json={
        "id": ESQL_TOOL_ID,
        "type": "esql",
        "description": (
            "Filter shoes by structured attributes. Use this when the user "
            "specifies a running surface, stability type, or shoe category."
        ),
        "configuration": {
            "query": (
                "FROM shoes "
                "| WHERE surface == ?surface AND stability == ?stability "
                "| KEEP sku, name, description, surface, stability, category "
                "| LIMIT 5"
            ),
            "params": {
                "surface": {
                    "type": "keyword",
                    "description": "Running surface: road or trail",
                },
                "stability": {
                    "type": "keyword",
                    "description": "Stability type: neutral or stability",
                },
            },
        },
    },
)
print("esql tool created")

In [ ]:
requests.delete(f"{KIBANA_URL}/api/agent_builder/agents/{AGENT_ID}", headers=headers)

r = requests.post(
    f"{KIBANA_URL}/api/agent_builder/agents",
    headers=headers,
    json={
        "id": AGENT_ID,
        "name": "Shoe Assistant",
        "description": (
            "Help shoppers identify a shoe or get a recommendation. Use "
            "search_shoes for open queries, filter_shoes when the user "
            "specifies surface or stability, and answer in one or two "
            "sentences."
        ),
        "configuration": {
            "tools": [{"tool_ids": [TOOL_ID, ESQL_TOOL_ID]}],
            "instructions": (
                "Use filter_shoes when the user mentions a surface (road, "
                "trail) or stability type. Use search_shoes for general "
                "queries. For identification queries return a single SKU. "
                "For recommendation queries return one SKU and a short reason."
            ),
        },
    },
)
r.raise_for_status()
print("Agent created")

In [ ]:
def converse(user_input: str) -> str:
    r = requests.post(
        f"{KIBANA_URL}/api/agent_builder/converse",
        headers=headers,
        json={"agent_id": AGENT_ID, "input": user_input},
    )
    r.raise_for_status()
    return r.json()


print("identify:", converse(Q_IDENTIFY))
print("recommend:", converse(Q_RECOMMEND))

## Pro tip: `named_queries` as a deterministic gate

Tag a narrow clause with `_name`. If `matched_queries` lists it, retrieval is confident and the LLM call can be skipped. Below, `name_match` only fires when at least three of the user's content words land in the product name.

In [ ]:
def gated_search(user_query: str) -> dict:
    body = {
        "query": {
            "bool": {
                "should": [
                    {
                        "match": {
                            "name": {
                                "query": user_query,
                                "minimum_should_match": 3,
                                "_name": "name_match",
                            }
                        }
                    },
                    {
                        "semantic": {
                            "field": "description_semantic",
                            "query": user_query,
                        }
                    },
                ]
            }
        },
        "size": 3,
        "_source": ["sku", "name"],
    }
    return es_client.search(index=INDEX_NAME, body=body)


def route(user_query: str) -> dict:
    res = gated_search(user_query)
    hits = res["hits"]["hits"]
    if hits and "name_match" in hits[0].get("matched_queries", []):
        return {
            "intent": "identify",
            "sku": hits[0]["_source"]["sku"],
            "source": "retrieval (gate fired)",
        }
    return {
        "intent": "recommend",
        "candidates": [h["_source"] for h in hits],
        "source": "forward to LLM",
    }


print("identify:", json.dumps(route("vaporfly 3 men's 10 black"), indent=2))
print("recommend:", json.dumps(route(Q_RECOMMEND), indent=2))

## Cleanup

In [ ]:
es_client.indices.delete(index=INDEX_NAME, ignore_unavailable=True)

requests.delete(f"{KIBANA_URL}/api/agent_builder/agents/{AGENT_ID}", headers=headers)
requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{TOOL_ID}", headers=headers)
requests.delete(f"{KIBANA_URL}/api/agent_builder/tools/{ESQL_TOOL_ID}", headers=headers)

try:
    es_client.inference.delete(inference_id="shoes-chat")
except Exception as e:
    print(f"shoes-chat: {e}")

print("Cleaned up")